# APEX AI — Machine Learning Models
**Calorie Prediction · Weight Change · Fitness Classification · Recommendation Engine**

This notebook trains four scikit-learn models on the APEX AI fitness dataset.
All models are saved as `.pkl` files for the FastAPI backend.

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ROOT      = Path('..') if Path('../datasets').exists() else Path('.')
DATA_DIR  = ROOT / 'datasets'
MODEL_DIR = ROOT / 'ai_models' / 'ml_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Setup complete')
print(f'   Data dir:  {DATA_DIR.resolve()}')
print(f'   Model dir: {MODEL_DIR.resolve()}')

## 2. Load Dataset

In [ ]:
# Load fitness profiles dataset
df = pd.read_csv(DATA_DIR / 'fitness_profiles.csv')
print(f'Shape: {df.shape}')
print(f'\nColumns:\n{df.dtypes}')
df.head()

In [ ]:
# Dataset statistics
print('=== Dataset Summary ===')
print(df.describe().round(2))

# Missing values check
missing = df.isnull().sum()
if missing.any():
    print(f'\n⚠️ Missing values:\n{missing[missing > 0]}')
else:
    print('\n✅ No missing values')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Age distribution
axes[0,0].hist(df['age'], bins=30, color='#4F86C6', edgecolor='white', linewidth=0.5)
axes[0,0].set_title('Age Distribution', fontweight='bold')
axes[0,0].set_xlabel('Age (years)')

# Weight distribution
axes[0,1].hist(df['weight_kg'], bins=30, color='#6BAA75', edgecolor='white', linewidth=0.5)
axes[0,1].set_title('Weight Distribution', fontweight='bold')
axes[0,1].set_xlabel('Weight (kg)')

# Calories (TDEE) distribution
axes[0,2].hist(df['calories_tdee'], bins=30, color='#E07A5F', edgecolor='white', linewidth=0.5)
axes[0,2].set_title('TDEE Distribution', fontweight='bold')
axes[0,2].set_xlabel('Calories (kcal/day)')

# Fitness level counts
level_counts = df['fitness_level'].value_counts()
axes[1,0].bar(level_counts.index, level_counts.values, color=['#4F86C6','#6BAA75','#E07A5F'])
axes[1,0].set_title('Fitness Level Distribution', fontweight='bold')

# Activity level vs calories
axes[1,1].scatter(df['activity_level'], df['calories_tdee'], alpha=0.3, color='#4F86C6', s=10)
axes[1,1].set_title('Activity Level vs TDEE', fontweight='bold')
axes[1,1].set_xlabel('Activity Level')
axes[1,1].set_ylabel('TDEE (kcal)')

# BMI distribution
bmi = df['weight_kg'] / (df['height_cm'] / 100) ** 2
axes[1,2].hist(bmi, bins=30, color='#9B72CF', edgecolor='white', linewidth=0.5)
axes[1,2].set_title('BMI Distribution', fontweight='bold')
axes[1,2].set_xlabel('BMI')

plt.suptitle('APEX AI — Fitness Dataset EDA', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'reports' / 'eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved')

## 4. Model 1 — Calorie Prediction (Linear Regression)

In [ ]:
FEATURES = ['age', 'weight_kg', 'height_cm', 'activity_level', 'gender']
TARGET   = 'calories_tdee'

X = df[FEATURES]
y = df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

calorie_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LinearRegression())
])
calorie_model.fit(X_tr, y_tr)

preds = calorie_model.predict(X_te)
mae   = mean_absolute_error(y_te, preds)
rmse  = mean_squared_error(y_te, preds, squared=False)
r2    = r2_score(y_te, preds)

print('📈 Calorie Model (Linear Regression)')
print(f'   MAE  : {mae:.1f} kcal')
print(f'   RMSE : {rmse:.1f} kcal')
print(f'   R²   : {r2:.4f}')

# Cross-validation
cv_scores = cross_val_score(calorie_model, X, y, cv=5, scoring='r2')
print(f'   CV R² (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_te, preds, alpha=0.3, s=15, color='#4F86C6')
mn, mx = y_te.min(), y_te.max()
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
plt.xlabel('Actual TDEE (kcal)')
plt.ylabel('Predicted TDEE (kcal)')
plt.title('Calorie Model — Actual vs Predicted', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# Save model
joblib.dump(calorie_model, MODEL_DIR / 'calorie_regression.pkl')
print(f'✅ Model saved → calorie_regression.pkl')

## 5. Model 2 — Weight Change Prediction (Random Forest)

In [ ]:
FEATURES_W = ['age', 'weight_kg', 'height_cm', 'activity_level', 'gender', 'calories_tdee']
TARGET_W   = 'weight_change_30d'

# Compute target if missing
if TARGET_W not in df.columns:
    df[TARGET_W] = ((df['calories_tdee'] - 2000) / 7700) * 30

X_w = df[FEATURES_W]
y_w = df[TARGET_W]
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_w, y_w, test_size=0.2, random_state=42)

weight_model = Pipeline([
    ('scaler', StandardScaler()),
    ('rf',     RandomForestRegressor(n_estimators=200, max_depth=8,
                                      min_samples_leaf=4, random_state=42, n_jobs=-1))
])
weight_model.fit(X_tr_w, y_tr_w)

preds_w = weight_model.predict(X_te_w)
print('⚖️ Weight Change Model (Random Forest)')
print(f'   MAE  : {mean_absolute_error(y_te_w, preds_w):.4f} kg')
print(f'   RMSE : {mean_squared_error(y_te_w, preds_w, squared=False):.4f} kg')
print(f'   R²   : {r2_score(y_te_w, preds_w):.4f}')

# Feature importance
feat_imp = pd.Series(
    weight_model.named_steps['rf'].feature_importances_,
    index=FEATURES_W
).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
feat_imp.plot(kind='barh', color='#6BAA75')
plt.title('Weight Model — Feature Importance', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

joblib.dump(weight_model, MODEL_DIR / 'weight_regression.pkl')
print('✅ Model saved → weight_regression.pkl')

## 6. Model 3 — Fitness Classification (Random Forest)

In [ ]:
FEATURES_C = ['age', 'weight_kg', 'height_cm', 'activity_level', 'gender']
TARGET_C   = 'fitness_level'

le = LabelEncoder()
y_c = le.fit_transform(df[TARGET_C])
X_c = df[FEATURES_C]
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2,
                                                     random_state=42, stratify=y_c)

fitness_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(n_estimators=300, max_depth=10,
                                       min_samples_leaf=3, random_state=42, n_jobs=-1))
])
fitness_clf.fit(X_tr_c, y_tr_c)

preds_c = fitness_clf.predict(X_te_c)
print('🏋️ Fitness Classifier (Random Forest)')
print(f'   Accuracy : {accuracy_score(y_te_c, preds_c):.4f}')
print()
print(classification_report(y_te_c, preds_c, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_te_c, preds_c)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Fitness Classifier — Confusion Matrix', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

joblib.dump(fitness_clf, MODEL_DIR / 'fitness_classifier.pkl')
joblib.dump({i: c for i, c in enumerate(le.classes_)},
            MODEL_DIR / 'fitness_classifier_labels.pkl')
print('✅ Model saved → fitness_classifier.pkl')

## 7. Model 4 — Recommendation Engine

In [ ]:
# Content-based exercise recommendation by fitness level
workout_df = pd.read_csv(DATA_DIR / 'workout_history.csv')
print(f'Workout history shape: {workout_df.shape}')
workout_df.head()

In [ ]:
# Build level-based recommendation lookup
def build_recommender(df):
    recs = {}
    for level_id, level_name in enumerate(['Beginner', 'Intermediate', 'Advanced']):
        if 'fitness_level' in df.columns:
            subset = df[df['fitness_level'] == level_name]
        else:
            subset = df  # use all if no level column

        top_exercises = (
            subset.groupby('exercise')
                  .agg(avg_sets=('sets', 'mean'),
                       avg_reps=('reps', 'mean'),
                       count=('exercise', 'count'))
                  .sort_values('count', ascending=False)
                  .head(10)
                  .reset_index()
        )

        recs[level_id] = [
            {
                "exercise":     row['exercise'],
                "avg_sets":     round(row['avg_sets']),
                "avg_reps":     round(row['avg_reps']),
                "difficulty":   level_id + 1,
                "muscle_group": "Various",
            }
            for _, row in top_exercises.iterrows()
        ]
        print(f'  {level_name}: {len(recs[level_id])} exercises recommended')
    return recs

recommender = build_recommender(workout_df)
joblib.dump(recommender, MODEL_DIR / 'recommender.pkl')
print('\n✅ Recommender saved → recommender.pkl')

## 8. Model Summary

In [ ]:
print('=' * 55)
print('  APEX AI ML Models — Training Complete')
print('=' * 55)
models_info = [
    ('calorie_regression.pkl',    'Linear Regression', 'TDEE prediction'),
    ('weight_regression.pkl',     'Random Forest',     '30-day weight change'),
    ('fitness_classifier.pkl',    'Random Forest',     'Fitness level (0/1/2)'),
    ('recommender.pkl',           'Content-based',     'Exercise recommendations'),
    ('fitness_classifier_labels.pkl', 'Label map',     'Level ID → name'),
]
for fname, mtype, task in models_info:
    path = MODEL_DIR / fname
    size = path.stat().st_size / 1024 if path.exists() else 0
    status = '✅' if path.exists() else '❌'
    print(f'  {status} {fname:<40} {size:6.1f} KB')
    print(f'     └─ {mtype} | {task}')
print('=' * 55)
print('Run: uvicorn backend.main:app --reload to start the API')